# Лабораторная работа 2: Метод Ньютона


## Метод Ньютона


## Затормозить тележку с реактивным двигателем


In [1]:
# библиотеки
import numpy as np

Рассматривается движение тележки с управлением
u(t)∈{0,1}, где:

u=1 — двигатель включён (идёт торможение),
u=0 — свободное движение.

Динамика системы:

- $\dot{x} = v$
- $\dot{v} = -au(t)$

Управление задаётся параметрами:

$$
{\tau} = (\tau_1, \tau_2, \tau_3, \tau_4)
$$

где:

1. $\tau_1$ — свободное движение
2. $\tau_2$ — торможение
3. $\tau_3$ — свободное движение (запрещённая зона)
4. $\tau_4$ — торможение

Оптимизируем:

$$
J(\tau) = \alpha (x_4 - x^*)^2 + \beta v_4^2 + \gamma \sum \tau_i
$$

С учётом штрафов:

$$
\tilde{J}(\tau) = J(\tau) + \mu \Phi(\tau)
$$

In [2]:
def sp(z, kappa):
    return (1.0 / kappa) * np.log(1 + np.exp(kappa * z))

def P(z, kappa):
    return sp(z, kappa) ** 2

In [3]:
def final_state(tau, params):
    tau1, tau2, tau3, tau4 = tau
    a, v0 = params["a"], params["v0"]

    x1 = v0 * tau1
    v1 = v0

    x2 = v0 * (tau1 + tau2) - 0.5 * a * tau2**2
    v2 = v0 - a * tau2

    x3 = x2 + v2 * tau3
    v3 = v2

    x4 = x3 + v3 * tau4 - 0.5 * a * tau4**2
    v4 = v3 - a * tau4

    return x2, x3, x4, v4

In [4]:
def J_tilde(tau, params):
    alpha, beta, gamma = params["alpha"], params["beta"], params["gamma"]
    mu, kappa = params["mu"], params["kappa"]
    xA, xB = params["xA"], params["xB"]
    xmax = params["x_star"]
    dmax = params["dmax"]

    x2, x3, x4, v4 = final_state(tau, params)

    # основной функционал
    J = alpha * (x4 - xmax)**2 + beta * v4**2 + gamma * np.sum(tau)

    # штрафы
    Phi = 0
    for t in tau:
        Phi += P(-t, kappa)

    Phi += P(tau[1] - dmax, kappa)
    Phi += P(tau[3] - dmax, kappa)

    Phi += P(x2 - xA, kappa)
    Phi += P(xB - x3, kappa)

    return J + mu * Phi

In [5]:
def grad(f, tau, params, h=1e-5):
    g = np.zeros_like(tau)
    for i in range(len(tau)):
        e = np.zeros_like(tau)
        e[i] = h
        g[i] = (f(tau + e, params) - f(tau - e, params)) / (2*h)
    return g

In [6]:
def hessian(f, tau, params, h=1e-4):
    n = len(tau)
    H = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            e_i = np.zeros(n)
            e_j = np.zeros(n)
            e_i[i] = h
            e_j[j] = h

            H[i, j] = (
                f(tau + e_i + e_j, params)
                - f(tau + e_i - e_j, params)
                - f(tau - e_i + e_j, params)
                + f(tau - e_i - e_j, params)
            ) / (4 * h**2)

    return H

In [7]:
def newton_method(f, tau0, params, max_iter=50):
    tau = tau0.copy()

    for k in range(max_iter):
        g = grad(f, tau, params)
        H = hessian(f, tau, params)

        # регуляризация
        H += 1e-6 * np.eye(len(tau))

        d = np.linalg.solve(H, -g)

        # Армихо
        alpha = 1.0
        c = 1e-4

        while f(tau + alpha*d, params) > f(tau, params) + c * alpha * np.dot(g, d):
            alpha *= 0.5

        tau = tau + alpha * d

        if np.linalg.norm(g) < 1e-6:
            break

    return tau

In [9]:
params = {
    "a": 4,
    "v0": 10,
    "xA": 8,
    "xB": 16,
    "x_star": 20.5,
    "dmax": 1.5,
    "alpha": 100,
    "beta": 100,
    "gamma": 1,
    "mu": 1e5,
    "kappa": 30
}

tau0 = np.array([1.0, 0.5, 1.0, 0.5])

opt_tau = newton_method(J_tilde, tau0, params)

print("Оптимальное τ:", opt_tau)

Оптимальное τ: [0.01928991 0.95751134 1.3194064  1.45164712]


C:\Users\Gwynbleidd\AppData\Local\Temp\ipykernel_31408\3552941656.py:2: RuntimeWarning: overflow encountered in exp
  return (1.0 / kappa) * np.log(1 + np.exp(kappa * z))
